# Notebook 05 — Session Management & Memory

## ทำไมต้องมี Session?

LINE OA เป็น **stateless protocol** — แต่ละ webhook event มาแยกกัน
Bot ต้องจำได้ว่าคุยเรื่องอะไรมาก่อน

**Solution:** Session Store เก็บ state ของแต่ละผู้ใช้

```
Request 1: user_id=U123, text="สวัสดี"
  → Load session(U123) → {} (ว่าง, ผู้ใช้ใหม่)
  → Process → state = {task_status: ACTIVE, user_goal: "ทักทาย"}
  → Save session(U123) = state

Request 2: user_id=U123, text="ต่อจากที่คุยกัน"
  → Load session(U123) → {task_status: ACTIVE, user_goal: "ทักทาย"}
  → Process (resume_router เห็นว่า ACTIVE!)
  → ...
```


## Session Model — `src/infra/session_model.py`

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional

# จำลอง session_model.py
# โค้ดจริง: src/infra/session_model.py

@dataclass
class Session:
    """
    Container สำหรับเก็บข้อมูลของ user แต่ละคน
    """
    user_id: str
    
    # LangGraph state (ที่ persist ระหว่าง requests)
    # ไม่รวม: messages, line_payload, line_messages (เหล่านี้ reset ทุก request)
    state: Dict[str, Any] = field(default_factory=dict)
    
    # Conversation history (LangChain messages)
    # จะถูก clear เมื่อ task เปลี่ยน COMPLETED → ACTIVE
    messages: List[Any] = field(default_factory=list)
    
    # ชื่อผู้ใช้ (ดึงจาก LINE Profile API)
    display_name: Optional[str] = None


# ตัวอย่าง session
session = Session(
    user_id="U1234567890",
    display_name="สมชาย",
    state={
        "conversation_mode": "AI",
        "task_status": "COMPLETED",
        "task_summaries": [
            {
                "task_id": "abc123",
                "user_goal": "ถามเรื่องที่จอดรถ",
                "outcome": "มีที่จอดรถ 500 คัน",
            }
        ]
    },
    messages=[]
)

print("Session object:")
print(f"  user_id: {session.user_id}")
print(f"  display_name: {session.display_name}")
print(f"  task_status: {session.state.get('task_status')}")
print(f"  task_summaries: {len(session.state.get('task_summaries', []))} items")
print(f"  messages: {len(session.messages)} messages")

## Session Store — Memory Backend

**ไฟล์:** `src/infra/session_store/memory.py`

ใช้สำหรับ development — เก็บใน RAM

In [ ]:
from typing import Optional

# จำลอง memory.py
# โค้ดจริง: src/infra/session_store/memory.py

_SESSION_STORE: dict = {}  # Simple in-memory dict

async def load_session_memory(user_id: str) -> Optional[Session]:
    """โหลด session จาก memory (None ถ้าไม่เจอ)"""
    return _SESSION_STORE.get(user_id)

async def save_session_memory(session: Session) -> None:
    """บันทึก session ลง memory"""
    _SESSION_STORE[session.user_id] = session


# ทดสอบ Memory Backend
import asyncio

async def test_memory_backend():
    user_id = "U_test_001"
    
    # Load ครั้งแรก → None
    s = await load_session_memory(user_id)
    print(f"Load new user: {s}")
    
    # สร้าง session ใหม่
    session = Session(user_id=user_id, display_name="Test User")
    session.state["task_status"] = "COMPLETED"
    
    # Save
    await save_session_memory(session)
    print(f"Saved session for {user_id}")
    
    # Load อีกครั้ง → ได้ session
    loaded = await load_session_memory(user_id)
    print(f"Loaded: user_id={loaded.user_id}, name={loaded.display_name}")
    print(f"  task_status: {loaded.state.get('task_status')}")
    
    # ⚠️ ข้อเสีย: restart server → หายหมด!
    print(f"\n⚠️  Memory store ไม่ persist — restart server = ข้อมูลหาย!")
    print("  → ใช้ Redis สำหรับ production")

asyncio.run(test_memory_backend())

## Session Store — Redis Backend

**ไฟล์:** `src/infra/session_store/redis.py`

ใช้สำหรับ production — persist ข้ามการ restart

In [ ]:
import json

# จำลอง redis.py (ไม่ต้อง run Redis จริง)
# โค้ดจริง: src/infra/session_store/redis.py

REDIS_PREFIX = "session"
REDIS_URL = "redis://localhost:6379/0"

def _serialize_session(session: Session) -> str:
    """
    แปลง Session เป็น JSON string สำหรับเก็บใน Redis
    
    ปัญหา: messages เป็น LangChain BaseMessage objects
    ต้อง serialize พิเศษด้วย message_to_dict()
    """
    # from langchain_core.messages import message_to_dict
    # serialized_messages = [message_to_dict(m) for m in session.messages]
    serialized_messages = []  # mock
    
    data = {
        "user_id": session.user_id,
        "display_name": session.display_name,
        "state": session.state,
        "messages": serialized_messages,
    }
    return json.dumps(data, ensure_ascii=False)


def _deserialize_session(data: str) -> Session:
    """
    แปลง JSON string กลับเป็น Session object
    """
    # from langchain_core.messages import messages_from_dict
    d = json.loads(data)
    # messages = messages_from_dict(d.get('messages', []))
    messages = []  # mock
    
    return Session(
        user_id=d['user_id'],
        display_name=d.get('display_name'),
        state=d.get('state', {}),
        messages=messages,
    )


async def load_session_redis(user_id: str) -> Optional[Session]:
    """
    โหลด session จาก Redis
    Key: f"session:{user_id}"
    """
    # async with redis.asyncio.from_url(REDIS_URL) as client:
    #     data = await client.get(f"{REDIS_PREFIX}:{user_id}")
    #     if data is None:
    #         return None
    #     return _deserialize_session(data)
    print(f"Redis GET key: {REDIS_PREFIX}:{user_id}")
    return None  # mock


async def save_session_redis(session: Session) -> None:
    """
    บันทึก session ลง Redis
    """
    # async with redis.asyncio.from_url(REDIS_URL) as client:
    #     await client.set(
    #         f"{REDIS_PREFIX}:{session.user_id}",
    #         _serialize_session(session)
    #     )
    serialized = _serialize_session(session)
    print(f"Redis SET key: {REDIS_PREFIX}:{session.user_id}")
    print(f"  Value (first 100 chars): {serialized[:100]}...")


# ทดสอบ serialization
test_session = Session(
    user_id="U_prod_001",
    display_name="สมชาย",
    state={
        "task_status": "COMPLETED",
        "task_summaries": [{"task_id": "t1", "user_goal": "ถามเรื่องที่จอดรถ"}]
    }
)

print("=== Redis Serialization ===")
serialized = _serialize_session(test_session)
print(f"Serialized JSON:\n{json.dumps(json.loads(serialized), ensure_ascii=False, indent=2)}")
print()

deserialized = _deserialize_session(serialized)
print(f"Deserialized: user_id={deserialized.user_id}, name={deserialized.display_name}")

## Message History Management

โปรเจคนี้จัดการ message history อย่างระมัดระวัง:

In [ ]:
# จำลองการจัดการ message history
# โค้ดจริง: src/integrations/line/handler_stub.py

class MockHumanMessage:
    def __init__(self, content): 
        self.content = content
        self.type = 'human'

class MockAIMessage:
    def __init__(self, content): 
        self.content = content  
        self.type = 'ai'


_MAX_MESSAGES_FOR_GRAPH = 50

def prepare_messages_for_graph(session: Session, user_text: str) -> list:
    """
    เตรียม messages list ก่อนส่งเข้า LangGraph
    
    1. ถ้า task_status เปลี่ยน COMPLETED → ACTIVE: clear messages
    2. เพิ่ม HumanMessage ใหม่
    3. ตัดเหลือแค่ 50 รายการล่าสุด
    """
    messages = list(session.messages)
    
    # Clear เมื่อเริ่ม task ใหม่
    current_status = session.state.get("task_status", "COMPLETED")
    if current_status == "COMPLETED" and user_text:
        messages = []  # เริ่ม conversation ใหม่
        print("  [clear messages - new task]")
    
    # เพิ่ม human message ใหม่
    if user_text:
        messages.append(MockHumanMessage(user_text))
    
    # ตัดเหลือแค่ 50 รายการล่าสุด
    return messages[-_MAX_MESSAGES_FOR_GRAPH:]


def save_ai_response(session: Session, final_answer: str):
    """
    บันทึก AI response เข้า message history
    เรียกหลังจากได้ state_out จาก LangGraph
    """
    if final_answer:
        session.messages.append(MockAIMessage(final_answer))


# Simulation
print("=== Message History Simulation ===")
print()

session = Session(user_id="U123", state={"task_status": "COMPLETED"})

# Request 1: New task
print("Request 1: 'สวัสดีครับ'")
msgs = prepare_messages_for_graph(session, "สวัสดีครับ")
print(f"  Messages to graph: {[m.content for m in msgs]}")
save_ai_response(session, "สวัสดีครับ! มีอะไรให้ช่วยไหม?")
session.state["task_status"] = "ACTIVE"  # simulate ACTIVE
session.messages = msgs + session.messages[-1:]  # sync
print(f"  Session messages: {len(session.messages)}")

print()

# Request 2: Continue active task
print("Request 2: 'ต่อเรื่องโรงพยาบาล' (task still ACTIVE)")
msgs2 = prepare_messages_for_graph(session, "ต่อเรื่องโรงพยาบาล")
print(f"  Messages to graph: {[m.content for m in msgs2]}")
session.state["task_status"] = "COMPLETED"  # task complete

print()

# Request 3: New task (after COMPLETED → will clear)
print("Request 3: 'เรื่องใหม่เลย' (after COMPLETED → clear!)")
msgs3 = prepare_messages_for_graph(session, "เรื่องใหม่เลย")
print(f"  Messages to graph: {[m.content for m in msgs3]}")

## Session Store Settings — Pluggable Backend

In [ ]:
import os

# จำลอง settings.py
# โค้ดจริง: src/infra/session_store/settings.py

SESSION_STORE_BACKEND = os.getenv("SESSION_STORE_BACKEND", "memory")

# Factory สำหรับเลือก backend
def get_session_store():
    if SESSION_STORE_BACKEND == "redis":
        return {
            "load": load_session_redis,
            "save": save_session_redis,
        }
    else:
        return {
            "load": load_session_memory,
            "save": save_session_memory,
        }

store = get_session_store()
print(f"Backend: {SESSION_STORE_BACKEND}")
print(f"Load function: {store['load'].__name__}")
print(f"Save function: {store['save'].__name__}")
print()

print("เปลี่ยน backend ด้วย ENV var:")
print("  SESSION_STORE_BACKEND=memory  → ใช้ in-memory dict (dev)")
print("  SESSION_STORE_BACKEND=redis   → ใช้ Redis (production)")
print()
print("Redis ENV vars:")
print("  REDIS_URL=redis://localhost:6379/0")
print("  REDIS_PREFIX=session  (key prefix = 'session:U123')")

## สรุป Session Architecture

```
Per-Request Flow:

┌─────────────────────────────────────────────────────────┐
│                  LINE Handler                           │
│                                                         │
│  load_session(user_id)                                  │
│       ↓                                                 │
│  [Memory: dict lookup]  OR  [Redis: GET key]           │
│       ↓                                                 │
│  Session = {state, messages, display_name}             │
│       ↓                                                 │
│  prepare messages (clear if new task + add HumanMsg)   │
│       ↓                                                 │
│  run_ai_runtime(session)  → LangGraph                  │
│       ↓                                                 │
│  save_ai_response(session, answer)                     │
│       ↓                                                 │
│  save_session(session)                                  │
│       ↓                                                 │
│  [Memory: dict update]  OR  [Redis: SET key JSON]      │
└─────────────────────────────────────────────────────────┘

State ที่ persist:
  ✓ conversation_mode, task_status, task_summaries
  ✓ route_module, current_agent_module
  ✗ messages (เก็บแยกใน session.messages)
  ✗ line_payload, line_messages (reset ทุก request)
```

**Notebook ถัดไป:** `06_end_to_end_flow.ipynb` — trace request ตั้งแต่ต้นจนจบ